In [0]:
from pathlib import Path
import pandas as pd
from pyspark.sql.functions import current_timestamp,regexp_replace, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

In [0]:
#Leemos la tabla en esquema bronze
df = spark.table("workspace.bronze.superstore")
df_p=df.toPandas()


In [0]:
df_p.info()

In [0]:
#Validamos que no hayan negativos en campo QUANTITY
df_p[df_p['Quantity'] < 0]

In [0]:
#Reemplazamos los espacios de las columnas con _ y renombramos con minusculas"
def clean_column_name(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace(".", "_").replace("-", "_")

df = df.toDF(*[clean_column_name(col) for col in df.columns])
df.columns

In [0]:
df.dtypes

In [0]:
display(df.head(5))

In [0]:

# Eliminamos row_id
df = df.drop("row_id")

# Reemplazamos las comas por puntos, para obtener un número decimal
df = df.withColumn("sales", regexp_replace(col("sales"), ",", ".").cast("float"))
df = df.withColumn("discount", regexp_replace(col("discount"), ",", ".").cast("float"))
df = df.withColumn("profit", regexp_replace(col("profit"), ",", ".").cast("float"))


In [0]:
#Guardamos la información en el esquema Silver
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
df.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.superstore")

In [0]:
display(df.head(5))

In [0]:
schema = StructType([
    StructField("nombre_capa", StringType(), True),
    StructField("nombre_tabla", StringType(), True),
    StructField("cnt_registros", IntegerType(), True)
])

df_control = spark.createDataFrame([
    ("silver", "superstore", int(df.count()))
], schema) \
    .withColumn("fecha_actualizacion", current_timestamp())


spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ctl")

# Creamos la tabla de control si no existe
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.ctl.control_procesos (
    nombre_capa STRING,
    nombre_tabla STRING,
    cnt_registros INT,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
""")

# Insertamos el registro en la tabla de control
df_control.write.format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('workspace.ctl.control_procesos')

display(df_control)